# పాఠం 12 - ఏజెంట్ స్క్రాచ్‌ప్యాడ్‌తో చాట్ చరిత్ర తగ్గింపు

ఈ నోట్‌బుక్ Microsoft Agent Framework ను ఉపయోగించి పొడవైన సంభాషణల్లో కాన్టెక్స్ట్‌ను ఎలా నిర్వహించాలో చూపిస్తుంది. సంభాషణలు పెరుగుతుండగా టోకెన్ సంఖ్య కూడా పెరుగుతుంది — చివరకు మోడల్ యొక్క కాన్టెక్స్ట్ విండోని మించి పోతుంది. దీనిని **కాంటెక్స్ట్ సారాంశ నమూనా** మరియు **ఏజెంట్ స్క్రాచ్‌ప్యాడ్** అనే శాశ్వత మెమరీతో పరిష్కరించాము.

## మీరు నేర్చుకునేది:
1. **ఎందుకు కాంటెక్స్ట్ నిర్వహణ ముఖ్యం**: టోకెన్ పరిమితులు మరియు కాంటెక్స్ట్ విండోలను అర్థం చేసుకోవడం
2. **కాంటెక్స్ట్-జ్ఞాత ఏజెంట్లు**: తమ స్వంత సంభాషణ కాంటెక్స్ట్‌ని నిర్వహించే ఏజెంట్లను నిర్మించడం
3. **కాంటెక్స్ట్ సారాంశ నమూనా**: సంభాషణ చరిత్రను సారాంశం చేయడానికి టూల్స్ ఉపయోగించడం
4. **ఏజెంట్ స్క్రాచ్‌ప్యాడ్**: కాంటెక్స్ట్ తగ్గింపును తట్టుకోవగల శాశ్వత మెమరీ

## అవసరమైన Voraussetzungen:
- Azure OpenAI సెటప్ కచ్చితంగా వాతావరణ మార్పులుతో కాన్ఫిగర్ చేయబడింది
- ముందుగా పాఠముల నుంచి ప్రాథమిక ఏజెంట్ కాన్సెప్ట్‌ల పూర్వగ్రహం


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## సందర్భం నిర్వహణ ఎందుకు ముఖ్యం

ప్రతి LLM కి ఒక పరిమిత **కాంటెಕ್ಸ్ట్ విండో** ఉంటుంది — ఒకే సారి ప్రాసెస్ చేయగల టోకెన్ల గరిష్ట సంఖ్య. బహుళ-తిళ్లు సంభాషణ కొనసాగుతూనే:

- ప్రతి వినియోగదారు సందేశం మరియు సహాయకుడు ప్రత్యుత్తరంతో **టోకెన్ లెక్క గణన రేఖాకారంగా పెరుగుతుంది**.
- **ప్రాంప్ట్ టోకెన్లు ఖర్చు ప్రధానంగా ఉంటాయి** ఎందుకంటే ప్రతి తిప్పలో మొత్తం చరిత్ర మళ్లీ పంపబడుతుంది.
- చివరికి సంభాషణ **కాంటెక్స్ట్ విండోను మించి పోతుంది** మరియు నమూనా లేదా తక్కువ చెయ్యబడుతుంది లేదా లోపం చూపిస్తుంది.

### సందర్భం నిర్వహణ వ్యూహాలు

| వ్యూహం | ఇది ఎలా పనిచేస్తుంది | వ్యతిరేకం |
|---|---|---|
| **తుంకడం** | పాత సందేశాలను తప్పించు | ప్రాథమిక సందర్భం కోల్పోతుంది |
| **సారాంశం చేయడం** | పాత సందేశాలను సారాంశం రూపంలో సంక్షిప్తం చేయండి | కొన్ని వివరాలు కోల్పోవచ్చు, కానీ ముఖ్యమైన అంశాలు నిలుపబడతాయి |
| **స్క్రాచ్ప్యాడ్ / బాహ్య మెమరీ** | సంభాషణ వెలుపల ముఖ్యమైన వాస్తవాలను నిల్వ చేయండి | టూల్ కాల్స్ అవసరం, కానీ ఏ తగ్గింపును ఎదుర్కొంటుంది |

ఈ నోట్‌బుక్‌లో మేము **సారాంశం చేయడం** మరియు **స్క్రాచ్ప్యాడ్ టూల్** ని కలిపి ఉపయోగిస్తాము, అందువల్ల ఏజెంట్ సంభాషణ చరిత్ర సంక్షిప్తమైనప్పటికీ నిరంతరత ఉంచగలుగుతాడు.


## సందర్భం-తెలిసిన ఏజెంట్ సృష్టించడం


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## దీర్ఘ సంభాషణను సిమ్యూలేట్ చేయడం

పరిస్థితులు ఎలా సేకరిస్తాయో చూడటానికి మల్టీ-టర్న్ సంభాషణను అనుసరించండి. ఏజెంట్ ముఖ్యమైన వివరాలు (ప్రాధాన్యతలు, బడ్జెట్, ప్రయాణ తేదీలు)ను టర్న్స్ గుండా నిలుపుకోవాలి మరియు సజావుగా కొనసాగింపు చూపించాలి.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ఏజెంట్ ముందు జరిగిన సంభాషణల నుంచి సందర్భాన్ని ఎలా ఉంచుకుంటుందో గమనించండి — ఇది జపాన్, సుషి, దేవాలయాలు, ఫొటోగ్రఫీ, $3000 బడ్జెట్, ఒంటరి ప్రయాణం, మరియు ఏప్రిల్ సమయం గురించి తెలుసుకొని ఉంటుంది. ఒక చిన్న సంభాషణలో ఇది బాగా పని చేస్తుంది, కానీ సంభాషణ పెరిగిన కొద్దీ పూర్తి చరిత్రను మళ్లీ పంపడం ఖర్చుతో జరుగుతుంది.

మరిన్ని సంభాషణ పుంతలతో సందర్భం ఎలా కూడుతున్నదో చూడడానికి సంభాషణ కొనసాగిద్దాం:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## పరిసర సంగ్రహ నమూనా

సంభాషణ పెరుగుతున్నప్పుడు, మేము సారాంశ పరికరాన్ని ఉపయోగించి సేకరించిన పరిసరాన్ని సంక్షిప్త ఫార్మాట్‌లో చురుకుగా మార్చవచ్చు. ఏజెంట్ ఈ పరికరాన్ని పిలిచి ముఖ్యమైన ఇష్టాలను రికార్డు చేస్తుంది, తద్వారా పాత సందేశాలు తొలగించినప్పటికీ మూల్యమైన సమాచారం నిల్వ ఉంటుంది.

ఈ నమూనా మరింత నైపుణ్యమైన చరిత్ర తగ్గింపులో నిర్మాణభాగం:
1. ఏజెంట్ సంభాషణ నుండి ముఖ్యమైన అంశాలు గుర్తిస్తుంది
2. అవి నిల్వ కోసం సారాంశ పరికరాన్ని పిలుస్తుంది
3. పాత సందేశాలను సురక్షితంగా తొలగించవచ్చు ఎందుకంటే సారాంశం ముఖ్యం అయిన విషయాలను పట్టుకుంటుంది

కింది భాగంలో, ఏజెంట్ తనకు తెలుసుకున్న విషయాలను సంక్షిప్తంగా రికార్డు చేయడానికి పిలుచుకోగల `summarize_preferences` పరికరాన్ని నిర్వచిస్తాం.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## సంగ్రహం

ఈ పాఠంలో మీరు Microsoft Agent Framework ఉపయోగించి దీర్ఘకాలిక ఏజెంట్ సంభాషణలలో సానుకూల సందర్భాలను ఎలా నిర్వహించాలో నేర్చుకున్నారు:

### ముఖ్యమైన భావనలు
- **సందర్భ కిటికీలు పరిమితం చేయబడ్డవి** — సంభాషణ చరిత్రలో ప్రతి టోకెన్ ఖర్చు చేస్తుంది మరియు పరిమితికి లెక్క చేయబడుతుంది.
- **సారాంశ సాధనాలు** ఏజెంట్ సుంకించిన సందర్భాన్ని సంక్షిప్త సారాంశాలకు మార్చడానికి అనుమతిస్తాయి, టోకెన్ వినియోగాన్ని తగ్గిస్తూ ముఖ్యమైన సమాచారాన్ని సంరక్షిస్తాయి.
- **ఏజెంట్ స్క్రాచ్ప్యాడ్లు** ఎటువంటి సంభాషణ తగ్గింపు కూడా సహించగల స్థిరంగా ఉన్న బాహ్య జ్ఞాపకాన్ని అందిస్తాయి.

### మీరు నిర్మించిన వాటి
- బహు-తొలగింపు సంభాషణలలో కొనసాగించునట్లుగా **సందర్భ-అవగాహన ఏజెంట్**
- ముఖ్య వినియోగదారు వివరాలను సంక్షిప్త రీతిలో రికార్డు చేసేందుకు **సారాంశ సాధనం** (`summarize_preferences`)
- సందర్భ నిలుపుదల మరియు మార్పు నిర్వహణను చూపించే **బహు-తొలగింపు సంభాషణ**

### వాస్తవ ప్రపంచ ఉపయోగాలు
- **కస్టమర్ సర్వీస్ బాట్లు**: దీర్ఘకాలం మద్దతు సत्रాలలో ప్రాధాన్యతలను గుర్తుంచుకోండి
- **వ్యక్తిగత సహాయకులు**: సందర్భం తిరిగి వివరిస్తున్నాడు లేకుండాప్రస్తుత ప్రాజెక్టులను ట్రాక్ చేయండి
- **విద్యా కళాశాలలు**: పలుసార్లు సంభాషణలలో విద్యార్థి పురోగతిని నిలుపుకోండి

### తదుపరి దశలు
- ఫైల్ ఆధారిత నిల్వతో పూర్తి స్క్రాచ్ప్యాడ్ సాధనాన్ని అమలు చేయండి
- సారాంశం తర్వాత స్వయంచాలక చరిత్ర తగ్గింపును జోడించండి
- సంధర్బ జ్ఞాపక శోధన కోసం వెక్టర్ డేటాబేస్‌లతో ఒకటి చేయండి
- సమగ్ర సందర్భంతో రోజులు తర్వాత సంభాషణలను పునఃప్రారంభించగల ఏజెంట్లను అభివృద్ధి చేయండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**విమర్శనాత్మక గుర్తింపు**:  
ఈ దస్త్రాన్ని AI అనువాద సేవ అయిన [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మనం సరిగ్గా అనువదించడానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో తప్పులొ అప్పుడు తప్పులొ ఉండవచ్చును‌ని దయచేసి గమనించండి. స్థానిక అసలు పత్రం అధికారిక మూలంగా పరిగణించాలి. అత్యంత ప్రాధాన్య సమాచారాన్ని పొందడానికి, ప్రొఫెషనల్ మానవ అనువాదం అవసరం. ఈ అనువాదం వలన కలిగే ఏమైనా అపార్థాలు లేదా తప్పుదోవకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
